# 17 Joint

Reads the safety and the linguistic results against each other. Both are
measured on the same requests, so the question this notebook asks is not whether
each moves with the stated age, which Sections 4.2 and 4.3 already answer, but
whether they move in the same shape and on the same scenarios.

Three analyses:

1. **The two ladders.** Refusal Rate and grade level at each of the eight stated
   ages, on one scale each.
2. **Threshold against gradient.** The ladder split into the movement across
   childhood, the step at the statutory boundary, and the movement above it, so
   the two outcomes can be compared as shares of their own range.
3. **Scenario concordance.** Whether the scenarios on which a model changes its
   safety behaviour are the scenarios on which it changes its reading level.

This notebook does not populate the predeclared Case C families of
`analysis.FAMILIES`. Every result below is descriptive, carries no adjusted
value, and is reported as such.

## Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

import analysis
import language
from analysis import (FOCUS, MACRO, NAME, ORDER, STATED, STATED_AGE,
                      by_scenario, publish, write_captions)

pd.set_option('display.width', 200, 'display.max_columns', 40)

AGES = [STATED_AGE[name] for name in STATED]
MINOR, ADULT = [a for a in AGES if a < 18], [a for a in AGES if a >= 18]
TABLES = Path.cwd().parent / 'tables'

In [ ]:
safety = analysis.load_corpus()
returned = safety[safety['responded']]

reading = language.load()
reading['label'] = reading['model'].map(NAME)
measurable = reading[reading['response_length'] >= language.FLOOR]
stated = measurable[measurable['signal'].eq('stated')]

print(f"{len(returned):,} returned replies, {len(stated):,} measurable at a stated age")

## 1. The two ladders

Refusal is read within Age Restricted scenarios, which is the stratum the
primary hypotheses are tested in and the only one whose expected answer moves
with age. Grade level is read over every stated-age reply, since it carries no
expectation to move.

In [ ]:
refusal = pd.DataFrame(
    {age: {model: by_scenario(returned[(returned['label'] == model)
                                       & returned['scenario_type'].eq(FOCUS)],
                              'refusal', [name]).mean() * 100
           for model in ORDER}
     for name, age in zip(STATED, AGES)}).reindex(ORDER)

# Scenario weighted, like the refusal ladder beside it: the scenario is the
# sampled unit, so a mean over replies would weight a scenario by how many of
# its replies cleared the word floor. Reply weighting reproduced neither
# Table 4.13 nor anything else, and differed by up to 0.17 grades on the models
# the floor costs most.
grade = pd.DataFrame(
    {age: (stated[stated['age'] == age]
           .groupby(['label', 'scenario_id'])['fkgl'].mean()
           .groupby('label').mean())
     for age in AGES}).reindex(ORDER)

ladders = pd.concat({'Refusal Rate (%)': refusal, 'Grade Level': grade},
                    names=['Measure', 'Model'])
ladders.columns = [f'Age {age}' for age in AGES]
publish(ladders.round(2), 'joint_01_ladders')
ladders.round(1)

## 2. Threshold against gradient

The ladder is split into three pieces: the movement across the six minor ages,
the single step from seventeen to eighteen, and the movement from eighteen to
twenty-one. The last column is the step as a share of the whole range, which is
what makes the two outcomes comparable despite their different units.

In [ ]:
# joint_01 above reproduces the published marginal ladders and is left alone.
# This asks a different question, where within the trajectory the movement sits,
# and for that each of the four ages entering the estimand must come from one
# scenario cohort. Otherwise a changing denominator is read as a change of
# shape: on the marginal ladder Gemini 3.5 Flash Lite draws age 7 from 20
# scenarios and age 21 from 25, because its blocking falls at the young end.
#
# Completeness is required at the four ages the decomposition actually uses and
# not at all eight, since requiring 9, 11, 13 and 15 would discard scenarios
# that no term here reads. The hierarchy is replicates, then the scenario-age
# mean, then the cohort restriction, then the age mean.
SHAPE_AGES = [7, 17, 18, 21]


def cohort_ladder(frame, column, scale):
    wide = frame.groupby(['scenario_id', 'age'])[column].mean().unstack()
    wide = wide.reindex(columns=SHAPE_AGES).dropna()
    return wide.mean() * scale, len(wide)


def shape(frames, column, scale, name):
    rows = {}
    for model in ORDER:
        row, complete = cohort_ladder(frames[model], column, scale)
        rows[model] = {'Across Childhood (7 to 17)': row[7] - row[17],
                       'Step at the Boundary (17 to 18)': row[17] - row[18],
                       'Above the Boundary (18 to 21)': row[18] - row[21],
                       'Full Range (7 to 21)': row[7] - row[21],
                       'Step as Share of Range (%)':
                           (row[17] - row[18]) / (row[7] - row[21]) * 100,
                       'n': complete}
    out = pd.DataFrame(rows).T.reindex(ORDER)
    out.loc[MACRO] = out.mean()
    # Macro-Average keeps the meaning it has everywhere else in the thesis, the
    # mean of the six model values, including in the share column. The ratio of
    # the macro-averaged step to the macro-averaged range is a different and
    # also useful quantity, so it takes its own row rather than overwriting a
    # cell whose label would then be wrong.
    # Every other cell on these two rows is a mean over models or a ratio of
    # two of them. A count is neither: there are not 145 age-restricted
    # scenarios, and a mean of six cohort sizes is not a cohort. Left empty.
    out.loc['Panel Ratio'] = np.nan
    out.loc[[MACRO, 'Panel Ratio'], 'n'] = np.nan
    out.loc['Panel Ratio', 'Step as Share of Range (%)'] = (
        out.loc[MACRO, 'Step at the Boundary (17 to 18)']
        / out.loc[MACRO, 'Full Range (7 to 21)'] * 100)
    out['n'] = out['n'].astype('Int64')
    out.index.name = 'Model'
    return pd.concat({name: out}, names=['Measure', 'Model'])


aged = returned.assign(age=lambda d: d['condition'].map(STATED_AGE)).dropna(subset=['age'])
shapes = pd.concat([
    shape({m: aged[(aged['label'] == m) & aged['scenario_type'].eq(FOCUS)] for m in ORDER},
          'refusal', 100, 'Refusal Rate (pp)'),
    shape({m: stated[stated['label'] == m] for m in ORDER},
          'fkgl', 1, 'Grade Level')])
publish(shapes.round(2), 'joint_02_shape')
shapes.round(1)


## 3. Scenario concordance

For each model and each scenario, the age-induced change in safety behaviour and
the age-induced change in reading level, both as stated minor ages against stated
adult ages. The question is whether the two rank together: if a model rewrites a
scenario for a child, does it also change what it is willing to do on that
scenario?

Spearman is used rather than Pearson because neither difference is expected to be
linear in the other, and the interval is a scenario bootstrap, resampling the
scenarios the correlation is computed over.

In [ ]:
# Direction. Both shifts are signed so that a larger positive value means
# stronger child-directed adaptation. Refusal rises for a stated minor, so
# safety is minor minus adult. Grade level falls for a stated minor, so reading
# is adult minus minor. Taking minor minus adult on both, as an earlier version
# did, made a positive rho mean that a larger safety movement went with *less*
# simplification, which is the opposite of the question being asked.
# A small stratum is flagged and still reported. Spearman is defined on any
# non-constant pair, and Age Restricted is the stratum this analysis exists
# for, so suppressing it at N = 17 would erase half the panel from the
# comparison that matters. Whether an interval can be formed at all is left
# to the bootstrap.
CAUTION = 20


def scenario_shift(frame, column, scale, invert):
    part = frame.groupby(['scenario_id', 'age'])[column].mean().unstack()
    part = part.reindex(columns=MINOR + ADULT).dropna()
    minor, adult = part[MINOR].mean(axis=1), part[ADULT].mean(axis=1)
    return ((adult - minor) if invert else (minor - adult)) * scale


def concordance(keep, draws, seed):
    """Spearman with a scenario bootstrap, or a reason it is not estimable."""
    if len(keep) < 3:
        return None, None, None, 'fewer than three scenarios'
    if keep['safety'].nunique() < 2:
        return None, None, None, 'safety shift constant'
    if keep['reading'].nunique() < 2:
        return None, None, None, 'reading shift constant'

    rng = np.random.default_rng(seed)
    rho = spearmanr(keep['safety'], keep['reading']).statistic
    drawn = []
    for _ in range(draws):
        sample = keep.iloc[rng.integers(0, len(keep), len(keep))]
        if sample['safety'].nunique() < 2 or sample['reading'].nunique() < 2:
            continue
        drawn.append(spearmanr(sample['safety'], sample['reading']).statistic)
    if len(drawn) < 0.95 * draws:
        return rho, None, None, f'bootstrap degenerate, {len(drawn)} of {draws} valid'
    low, high = np.percentile(drawn, [2.5, 97.5])
    note = 'estimated' if len(keep) >= CAUTION else 'small stratum, interpret cautiously'
    return rho, low, high, note


types = returned.drop_duplicates('scenario_id').set_index('scenario_id')['scenario_type']
rows = []
for model in ORDER:
    left = returned[returned['label'] == model].assign(
        age=lambda d: d['condition'].map(STATED_AGE)).dropna(subset=['age'])
    right = stated[stated['label'] == model]
    both = pd.concat({'safety': scenario_shift(left, 'refusal', 100, False),
                      'reading': scenario_shift(right, 'fkgl', 1, True)},
                     axis=1).dropna()
    for stratum in ['Pooled'] + list(analysis.STRATA):
        keep = both if stratum == 'Pooled' else both[types.reindex(both.index).eq(stratum)]
        rho, low, high, reason = concordance(keep, analysis.DRAWS, analysis.SEED)
        rows.append({'Model': model, 'Scenario Type': stratum, 'n': len(keep),
                     'rho': rho, '95% CI Lower': low, '95% CI Upper': high,
                     'Reason': reason})

concordance_table = pd.DataFrame(rows).set_index(['Scenario Type', 'Model'])
publish(concordance_table.round(3), 'joint_03_concordance')
concordance_table.round(2)


In [ ]:
write_captions()


## What this notebook writes

| Table | Tier |
|---|---|
| `joint_01_ladders` | main |
| `joint_02_shape` | main |
| `joint_03_concordance` | supplement |

All three are descriptive. None declares a family, none carries an adjusted
value, and Section 4.4 reports them as description.

All three go through `publish()`, which reads the tier and the caption from
`config/captions.yml` and refuses a name that file does not describe.
`write_captions()` then merges the rendered entries into
`tables/captions.csv`, which is generated output rather than a source.